# AIC — Notebook 02: OCR Extraction

Runs PaddleOCR on all keyframe images and saves per-video JSON files.

**Input:** Kaggle dataset keyframe images
**Output:** `/kaggle/working/ocr/L{XX}_{V}.json`

**Estimated time:** ~2-4 hours for full dataset on T4 GPU

In [ ]:
import subprocess, sys, os
GITHUB_REPO = "https://github.com/YOUR_USERNAME/AIC_System.git"
REPO_DIR = "/kaggle/working/AIC_System"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git","clone","--depth","1",GITHUB_REPO,REPO_DIR],check=True)
else:
    subprocess.run(["git","-C",REPO_DIR,"pull"],check=True)
sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",f"{REPO_DIR}/requirements.txt"],check=True)
print('Setup complete.')

In [ ]:
from pathlib import Path
DATASET_SLUG = "your-username/aic-hcmc-data"  # ← change this
DATASET_NAME = DATASET_SLUG.split('/')[-1]
DATASET_PATH = Path(f"/kaggle/input/{DATASET_NAME}")
MAP_KF_DIR   = DATASET_PATH / "map-keyframes-aic25-b1" / "map-keyframes"
KF_IMG_ROOT  = DATASET_PATH / "keyframes" / "keyframes"
OUTPUT_DIR   = Path("/kaggle/working/ocr")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
csv_files = sorted(MAP_KF_DIR.glob("*.csv"))
print(f'Found {len(csv_files)} videos to process')

In [ ]:
from src.feature_extractors.ocr_extractor import OCRExtractor
extractor = OCRExtractor(lang='vi', min_confidence=0.6, use_gpu=True)
extractor.load()
print('OCR extractor ready.')

In [ ]:
from tqdm import tqdm
errors = []
for csv_path in tqdm(csv_files, desc='OCR Extraction'):
    video_id = csv_path.stem
    try:
        extractor.extract_video(
            video_id=video_id,
            keyframes_dir=str(KF_IMG_ROOT),
            map_keyframes_csv=str(csv_path),
            output_dir=str(OUTPUT_DIR),
            overwrite=False,
        )
    except Exception as e:
        errors.append((video_id, str(e)))
        print(f'ERROR: {video_id}: {e}')
print(f'Done. Errors: {len(errors)}')
print(f'Output files: {len(list(OUTPUT_DIR.glob("*.json")))}')
if errors:
    print('Failed videos:', [v for v,_ in errors])

In [ ]:
# Show stats
import json
total_kf, total_with_text = 0, 0
for f in OUTPUT_DIR.glob('*.json'):
    d = json.load(open(f))
    for kf in d.get('keyframes',[]):
        total_kf += 1
        if kf.get('texts'): total_with_text += 1
print(f'Total keyframes processed: {total_kf:,}')
print(f'Keyframes with OCR text:   {total_with_text:,} ({100*total_with_text/max(total_kf,1):.1f}%)')